## <a href="https://cursos.alura.com.br/course/langchain-desenvolva-agentes-inteligencia-artificial/task/161395?b2cUser=true"><b>Langchain Agentes - Refatorando</b></a><br/>

<b>Objetivo:</b> Criação do agente que chamará e executará a ferramenta que busca dados de estudante em um arquivo CSV.<br/>
<ul><li>Maneira pela qual a LLM toma ciência e executa a ferramenta.</li></ul>

<b>PASSOS:</b><br/>
<ul>
    <b><li>CRIAÇÃO DAS FERRAMENTAS</li></b><br/>
    <ul>
        <ol>
            <li>Criação da Ferramenta DadosDeEstudante (Refinando a anterior)</li>
            <li>Instanciando a Ferramenta que a LLM precisa usar</li>
        </ol>
    </ul><br/>
    <b><li>CRIAÇÃO DO AGENTE DE FERRAMENTAS</li></b><br/>   
    <ul>
        <ol>
            <li>Informando para a LLM as ferramentas que eu tenho (Usa a ferramenta que foi instanciada)</li>
            <li>Executando o Agente com a ferramenta</li>
        </ol>
    </ul><br/>
</ul>

In [1]:
#%pip install -r requirements.txt

In [2]:
from pydantic import BaseModel, Field

#from langchain.globals import set_debug

#set_debug(True)

class ExtratordeEstudante(BaseModel):
    estudante: str = Field(description="Nome do estudante informado, sempre em letras minúsculas. Exemplo: joão, carla, joana")

### <b>CRIAÇÃO DE FERRAMENTAS</b>
Que ferramentas eu tenho disponíveis para se obter os dados da Ana ?

<b>1) Criação da Ferramenta DadosDeEstudante</b>
<ul>
    <li> A classe deve estender de BaseTool</li>
    <li> Essa ferramenta deve ter nome, descrição e o método run estendido de BaseTool</li>    
</ul>

In [3]:
from langchain.tools import BaseTool
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
from pandas import read_csv

class DadosDeEstudante(BaseTool): # ESTENDE BaseTool
    
    llm:ChatOpenAI = None
    
    # TODA FERRAMENTA PRECISA TER ESSES ATRIBUTOS
    name: str = "dados_de_estudante" # Nome da ferramenta
    description : str = """ 
                            Essa ferramenta extrai o histórico e preferências de um estudante, de acordo com o seu histórico.
                        """ # Descrição da ferramenta    
    
    def __init__(self,llm:ChatOpenAI):
        super().__init__() # PARA NÃO SOBRESCREVER O CONSTRUTOR DA CLASSE MÃE BaseTool
        self.llm = llm        
    
    def __busca_dados_de_estudante(self,estudante:str) -> str:
        
        dfestudantes = read_csv("documentos/estudantes.csv")
        
        #print('Estudante buscado:', estudante)
        
        dados_estudante = dfestudantes.loc[dfestudantes['USUARIO'] == estudante]
        
        if dados_estudante.empty:
            return f"Desculpe, não encontrei dados para o estudante '{estudante}'. Por favor, verifique o nome e tente novamente."
        
        dict = dados_estudante.to_dict(orient='records')[0]
        
        print('Dicionário retornado pela ferramenta DadosDeEstudante:', dict)        
        
        return dict
    
                
    # CONTRATO DA FERRAMENTA - O QUE ELA FAZ
    def _run(self, input: str) -> str:        
        
        parseador = JsonOutputParser(pydantic_object=ExtratordeEstudante)    
        
        template = PromptTemplate(
                                    template = """ 
                                                    Você deve analisar a {input} e extrair o nome de estudante informado.
                                            
                                                    FORMATO DE SAIDA:
                                                    {formato_saida} 
                                                """,
                                    input_variables = ["input"],
                                    partial_variables = {"formato_saida": parseador.get_format_instructions()}
                                 )
        
        cadeia = template | self.llm | parseador
        
        resposta = cadeia.invoke({"input": input})
        
        estudante = resposta['estudante']
        
        print('Retorno estudante da LLM:', estudante)
        
        return self.__busca_dados_de_estudante(estudante)


<b>2) Instanciando a Ferramenta que a LLM precisa usar</b>
<ul>
    <li> Para o objeto da classe Tool, deverão ser informados o nome, a função, que é o método run da ferramenta que foi criada, e a descrição</li>
</ul>

In [4]:
from langchain.agents import Tool

class Tools:
        
        def __init__(self,llm:ChatOpenAI):
                
                dados_de_estudante = DadosDeEstudante(llm) # INSTANCIANDO O OBJETO DA MINHA FERRAMENTA

                # MATRIZ DE FERRAMENTAS (CONJUNTO DE FERRAMENTAS)
                self.tools = [
                        # Instanciando ferramentas
                        Tool(
                                name=dados_de_estudante.name,
                                func=dados_de_estudante.run,
                                description=dados_de_estudante.description                
                        )
                ]

### <b>CRIAÇÃO DO AGENTE DE FERRAMENTAS</b>

<b>3) Informando para a LLM as ferramentas que eu tenho</b> 
<ul><li>Para isso, é necessário criar um agente com as ferramentas</li></ul>

O Refactoring sempre deve iniciar pela classe final do código, que é a classe que contém todas as outras

In [5]:
from langchain.agents import create_openai_tools_agent
from langchain import hub
from dotenv import load_dotenv
from os import getenv
import warnings

warnings.filterwarnings("ignore")

class AgenteOpenAIFunctions:
    
    def __init__(self):
      
      load_dotenv()

      llm = ChatOpenAI(
                        model="gpt-5-mini",
                        api_key=getenv("API_KEY")            
                      )
      
      # INSTANCIANDO AS FERRAMENTAS
      self.tools = Tools(llm).tools
        
      # PROMPT DE INICIALIZAÇÃO PARA INFORMAR PARA A LLM SOBRE A FERRAMENTA.
      prompt=(hub.pull(owner_repo_commit="hwchase17/openai-functions-agent"))

      # CRIANDO UM AGENTE COM AS FERRAMENTAS
      self.agente = create_openai_tools_agent(
                                                llm=llm, # INFORMA A LLM QUE VAI SER USADA PELO AGENTE
                                                tools=self.tools, # PASSANDO PARA A LLM A FERRAMENTA QUE ELA PODE USAR. INSTÂNCIA DA FERRAMENTA
                                                prompt=prompt  
                                                                    # JÁ EXISTEM PROMPTS PRONTOS NO REPOSITÓRIO DO LANGSMITH, DE ACORDO COM O TIPO DE FERRAMENTA. 
                                                                          # Para agente de função (https://smith.langchain.com/hub/hwchase17/openai-functions-agent)
                                                                                
                                             )

      print(prompt)


<b>4) Executando o agente com a ferramenta</b>

Pode-se até passar dois nomes de uma vez.

In [ ]:
from langchain.agents import AgentExecutor

agente = AgenteOpenAIFunctions()

executor = AgentExecutor(
                            agent=agente.agente, # O AGENTE QUE VAI SER USADO
                            tools=agente.tools, # FERRAMENTAS QUE O AGENTE PODE USAR
                            verbose=True
                        )

perguntas = [ 
                "Quais são os dados da Ana e da Bianca?",
                "Crie um perfil acadêmico para a Ana." # ASSUNTO PARA O PRÓXIMO TÓPICO, AONDE DESEJO FORMATAR O PERFIL ACADÊMICO, A PARTIR 
                                                       # DOS DADOS DE ESTUDANTE RETORNADOS. A FERRAMENTA NÃO CONSEGUE RECEBER ESSES DADOS SOZINHA.       
           ]

for pergunta in perguntas:
    print('Pergunta: ', pergunta)
    resposta = executor.invoke({"input": pergunta})
    print(resposta)

input_variables=['agent_scratchpad', 'input'] optional_variables=['chat_history'] input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')]